In [1]:
!pip install ninja

In [2]:
!nvidia-smi

Tue Jul 14 14:09:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

In [17]:
cuda_source = """
#include <cuda_runtime.h>
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
__global__ void gelu_kernel(
    const float* input, float* output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        float x = input[idx];
        output[idx] = 0.5f * x * (1.0f + tanhf(0.7978845608f * (x + 0.044715f * x * x * x)));
    }
}
torch::Tensor gelu_forward(torch::Tensor input) {
    auto output = torch::empty_like(input);
    int N = input.numel();
    int threads = 256;
    int blocks = (N + threads - 1) / threads;
    
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();
    gelu_kernel<<<blocks, threads, 0, stream>>>(input.data_ptr<float>(), output.data_ptr<float>(), N);
    return output;}
"""

In [18]:
cpp_source = "torch::Tensor gelu_forward(torch::Tensor input);"

In [19]:
module = load_inline(
    name="gelu_streams",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["gelu_forward"],
    verbose=True
)

N = 1 << 24
CHUNKS = 8

In [20]:
x = torch.randn(N, device='cuda', dtype=torch.float32)
y = module.gelu_forward(x)

In [24]:
ref = F.gelu(x, approximate="tanh")
print(torch.allclose(y, ref, atol=1e-5)) 

True
